# Advanced Reinforcement Learning Project - Experiment 2

In [ ]:
!pip install gymnasium[box2d] swig renderlab torch matplotlib moviepy==1.0.3 opencv-python

In [ ]:
import gymnasium as gym
from gymnasium.vector import AsyncVectorEnv
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Normal
from torch.optim import AdamW
import matplotlib.pyplot as plt
from collections import namedtuple, deque
import random
import copy
import warnings
import renderlab as rl
warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Hyperparameters

In [ ]:
LR = 1e-4
BATCH_SIZE = 64
GAMMA = 0.99
GAE_LAMBDA = 0.95
CLIP_EPS = 0.1
N_EPOCHS = 10
N_ENVS = 10
N_STEPS = 2048
HIDDEN_SIZE = 128
BUFFER_SIZE = 1000000
MIN_REPLAY_SIZE = 10000
TAU = 0.005
ALPHA = 0.05

## On-Policy Agent: PPO-GAE

In [ ]:
class ActorNet(nn.Module):
    def __init__(self, input_size, hidden_units, output_size):
        super(ActorNet, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_size, hidden_units),
            nn.Tanh(),
            nn.Linear(hidden_units, int(hidden_units/2)),
            nn.Tanh()
        )
        self.mu_head = nn.Linear(int(hidden_units/2), output_size)
        self.logstd_head = nn.Linear(int(hidden_units/2), output_size)

    def forward(self, x):
        x = self.model(x)
        # LunarLander is [-1, 1], loc isn't multiplied by 2
        loc = torch.tanh(self.mu_head(x)) 
        scale = torch.exp(self.logstd_head(x))
        return loc, scale

class CriticNet(nn.Module):
    def __init__(self, input_size, hidden_units):
        super(CriticNet, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_size, hidden_units),
            nn.Tanh(),
            nn.Linear(hidden_units, int(hidden_units/2)),
            nn.Tanh()
        )
        self.value_head = nn.Linear(int(hidden_units/2), 1)

    def forward(self, x):
        x = self.model(x)
        value = self.value_head(x)
        return value

    # Call function was redundant as Python automatically provides it when inheriting nn.Module   


class RunningMem:
    def __init__(self):
        self.reset()

    def store(self, obs, action, logprob, reward, done, values):
        # Simplified store for this implementation structure, matching internal logic
        self.obs.append(obs)
        self.actions.append(action)
        self.logprobs.append(logprob)
        self.rewards.append(reward)
        self.dones.append(done)
        self.values.append(values)

    def reset(self):
        self.obs = []
        self.actions = []
        self.logprobs = []
        self.rewards = []
        self.dones = []
        self.values = []

    # def batches was integrated into train_ppo instead of its own function

In [ ]:
# Function used to run PPO training Loop inspired by code snippets from PPO-GAE2, parallel env, continuous (Pendulum).
def train_ppo(params, name="PPO", seed=50, max_steps=1000000):
    # Unpacks hyperparameters
    batch_size = params.get('BATCH_SIZE', BATCH_SIZE)
    gamma = params.get('GAMMA', GAMMA)
    gae_lambda = params.get('GAE_LAMBDA', GAE_LAMBDA)
    n_epochs = params.get('N_EPOCHS', N_EPOCHS)
    n_envs = params.get('N_ENVS', N_ENVS)
    n_steps = params.get('N_STEPS', N_STEPS)
    hidden_size = params.get('HIDDEN_SIZE', HIDDEN_SIZE)
    lr = params.get('LR', LR)
    clips = params.get('CLIP_EPS', CLIP_EPS)

    # Sets seeds for reproducibility
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

    # Added code for parallel envirements to speed up processing
    env_name = "LunarLanderContinuous-v3"
    def make_env():
        return gym.make(env_name)
    
    envs = AsyncVectorEnv([make_env for _ in range(n_envs)], seed=seed)
    
    obs_dim = envs.single_observation_space.shape[0]
    act_dim = envs.single_action_space.shape[0]

    actor = ActorNet(obs_dim, hidden_size, act_dim).to(device)
    critic = CriticNet(obs_dim, hidden_size).to(device)
    
    opt_actor = AdamW(actor.parameters(), lr=lr)
    opt_critic = AdamW(critic.parameters(), lr=lr)

    obs, _ = envs.reset()
    mem = RunningMem()
    
    ep_rewards = deque(maxlen=50)
    running_rewards = np.zeros(n_envs)
    
    # Tracks for logging
    tracker = {'steps': [], 'avg_reward': []}
    step_count = 0
    
    print(f"[{name}] Training PPO... LR={lr} | HIDDEN={hidden_size} | CLIPS={clips}")

    while step_count < max_steps:
        for _ in range(n_steps):
            step_count += n_envs
            
            obs_t = torch.as_tensor(obs, dtype=torch.float32).to(device)
            # Simulates actions
            with torch.no_grad():
                loc, scale = actor(obs_t)
                dist = Normal(loc, scale + 1e-6)
                action = dist.sample()
                log_prob = dist.log_prob(action).sum(axis=-1)
                value = critic(obs_t).squeeze()

            next_obs, rewards, terminations, truncations, _ = envs.step(action.cpu().numpy())
            dones = terminations | truncations

            mem.store(obs_t, action, log_prob, 
                      torch.as_tensor(rewards, dtype=torch.float32).to(device),
                      torch.as_tensor(dones, dtype=torch.float32).to(device),
                      value)
            
            running_rewards += rewards
            for i, done in enumerate(dones):
                if done:
                    ep_rewards.append(running_rewards[i])
                    running_rewards[i] = 0

            obs = next_obs
        
        # Used the batches function from PPO-GAE2
        # Computes Advantages (GAE)
        with torch.no_grad():
            next_value = critic(torch.as_tensor(next_obs, dtype=torch.float32).to(device)).squeeze()
        
        b_obs = torch.stack(mem.obs)
        b_actions = torch.stack(mem.actions)
        b_logprobs = torch.stack(mem.logprobs)
        b_rewards = torch.stack(mem.rewards)
        b_dones = torch.stack(mem.dones)
        b_values = torch.stack(mem.values)

        # Computes advantages in reverse order
        advantages = torch.zeros_like(b_rewards).to(device)
        lastgaelam = 0
        for t in reversed(range(n_steps)):
            # Handles terminal vs non-terminal states
            if t == n_steps - 1:
                nextnonterminal = 1.0 - b_dones[t]
                nextvalues = next_value  # Uses bootstrap value for last step
            else:
                nextnonterminal = 1.0 - b_dones[t]
                nextvalues = b_values[t+1]
            
            # TD error: reward + discounted next value - current value
            delta = b_rewards[t] + gamma * nextvalues * nextnonterminal - b_values[t]
            # Accumulates discounted advantages
            advantages[t] = lastgaelam = delta + gamma * gae_lambda * nextnonterminal * lastgaelam

        returns = advantages + b_values

        b_obs = b_obs.view(-1, obs_dim)
        b_actions = b_actions.view(-1, act_dim)
        b_logprobs = b_logprobs.view(-1)
        b_returns = returns.view(-1)
        b_advantages = advantages.view(-1)

        # Optimization
        b_inds = np.arange(n_envs * n_steps)
        for _ in range(n_epochs):
            np.random.shuffle(b_inds)
            for start in range(0, len(b_inds), batch_size):
                end = start + batch_size
                mb_inds = b_inds[start:end]

                new_loc, new_scale = actor(b_obs[mb_inds])
                new_dist = Normal(new_loc, new_scale + 1e-6)
                new_logprob = new_dist.log_prob(b_actions[mb_inds]).sum(axis=-1)
                
                ratio = torch.exp(new_logprob - b_logprobs[mb_inds])
                mb_advantages = b_advantages[mb_inds]
                mb_advantages = (mb_advantages - mb_advantages.mean()) / (mb_advantages.std() + 1e-8)

                surr1 = ratio * mb_advantages
                surr2 = torch.clamp(ratio, 1.0 - clips, 1.0 + clips) * mb_advantages
                actor_loss = -torch.min(surr1, surr2).mean()
                
                new_value = critic(b_obs[mb_inds]).squeeze()
                critic_loss = F.mse_loss(new_value, b_returns[mb_inds])

                opt_actor.zero_grad()
                actor_loss.backward()
                opt_actor.step()

                opt_critic.zero_grad()
                critic_loss.backward()
                opt_critic.step()

        mem.reset()

        # Logging
        if len(ep_rewards) > 0:
            avg_r = np.mean(ep_rewards)
            tracker['steps'].append(step_count)
            tracker['avg_reward'].append(avg_r)
            
            if step_count % 50000 < n_envs * n_steps:
                print(f"{name} Step: {step_count} | Avg Reward: {avg_r:.2f}")
            
            if len(ep_rewards) == 50 and avg_r >= 195.0:
                print(f"{name} Step: {step_count} | Avg Reward(50ep): {avg_r:.2f}")
                break
                
    envs.close()
    return tracker, actor

## Off-Policy Agent: SAC

In [ ]:
class PolicyNet(nn.Module):
    def __init__(self, input_size, hidden_units, output_size, pmin=None, pmax=None):
        super(PolicyNet, self).__init__()
        self.pmin = pmin
        self.pmax = pmax
        
        self.model = nn.Sequential(
            nn.Linear(input_size, hidden_units),
            nn.ReLU(),
            nn.Linear(hidden_units, int(hidden_units/2)),
            nn.ReLU()
        )
        self.mu_head = nn.Linear(int(hidden_units/2), output_size)
        self.logstd_head = nn.Linear(int(hidden_units/2), output_size)

    def forward(self, x):
        x = self.model(x)
        
        mu = self.mu_head(x)
        log_std = self.logstd_head(x)

        log_std = torch.clamp(log_std, -20, 2)
        std = torch.exp(log_std)
        
        dist = Normal(mu, std)
        u = dist.rsample()
        

        action_tanh = torch.tanh(u)
        # For LunarLander, pmax is 1.0
        action = action_tanh * self.pmax if self.pmax else action_tanh
        
        log_prob = dist.log_prob(u)
        log_prob -= 2 * (np.log(2) - u - F.softplus(-2*u))
        
        if self.pmax:
             log_prob -= np.log(self.pmax)

        log_prob = log_prob.sum(dim=-1, keepdim=True)
        return action, log_prob

class DQN(nn.Module):
    def __init__(self, input_size, hidden_units):
        super(DQN, self).__init__()
        self.Q = nn.Sequential(
            nn.Linear(input_size, hidden_units),
            nn.ReLU(),
            nn.Linear(hidden_units, int(hidden_units/2)),
            nn.ReLU(),
            nn.Linear(int(hidden_units/2), 1)
        )

    def forward(self, state, action):
        x = torch.cat([state, action], 1)
        q = self.Q(x)
        return q

Transition = namedtuple('Transition', ('states', 'actions', 'rewards', 'dones', 'next_states'))

class Replay_memory():
    def __init__(self, env, fullsize, minsize, batchsize):
        self.env = env
        self.memory = deque(maxlen=fullsize)
        self.rewards = deque(maxlen=50)
        self.batchsize = batchsize
        self.minsize = minsize

    def append(self, transition):
        self.memory.append(transition)

    def sample_batch(self):
        batch = random.sample(self.memory, self.batchsize)
        batch = Transition(*zip(*batch))
        return (
            torch.tensor(np.array(batch.states), dtype=torch.float32).to(device),
            torch.tensor(np.array(batch.actions), dtype=torch.float32).to(device),
            torch.tensor(np.array(batch.rewards), dtype=torch.float32).unsqueeze(1).to(device),
            torch.tensor(np.array(batch.dones), dtype=torch.float32).unsqueeze(1).to(device),
            torch.tensor(np.array(batch.next_states), dtype=torch.float32).to(device)
        )

In [ ]:
# Function used to run SAC training, inspired by Sac (Pendulum).
def train_sac(params, name="SAC", seed=50, max_steps=300000):

    # Unpacks hyperparameters
    gamma = params.get('GAMMA', GAMMA)
    buffer_size = params.get('BUFFER_SIZE', BUFFER_SIZE)
    min_replay_size = params.get('MIN_REPLAY_SIZE', MIN_REPLAY_SIZE)
    tau = params.get('TAU', TAU)
    n_envs = params.get('N_ENVS', N_ENVS)
    hidden_size = params.get('HIDDEN_SIZE', HIDDEN_SIZE)
    lr = params.get('LR', LR)
    alpha = params.get('ALPHA', ALPHA)

    # Sets seeds for reproducibility
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

    # Used to create multiple environments for parallel training
    env_name = "LunarLanderContinuous-v3"
    def make_env():
        return gym.make(env_name)
    
    envs = AsyncVectorEnv([make_env for _ in range(n_envs)], seed=seed)
    
    replay_memory = Replay_memory(None, buffer_size, min_replay_size, params['BATCH_SIZE'])

    obs_dim = envs.single_observation_space.shape[0]
    act_dim = envs.single_action_space.shape[0]

    # pmin/pmax for LunarLander (-1, 1)
    pmin = float(envs.single_action_space.low[0])
    pmax = float(envs.single_action_space.high[0])

    actor = PolicyNet(obs_dim, hidden_size, act_dim, pmin, pmax).to(device)
    q_net1 = DQN(obs_dim + act_dim, hidden_size).to(device)
    q_net2 = DQN(obs_dim + act_dim, hidden_size).to(device)
    
    q_net1_target = copy.deepcopy(q_net1)
    q_net2_target = copy.deepcopy(q_net2)

    q_net_optimizer = AdamW(list(q_net1.parameters()) + list(q_net2.parameters()), lr=lr)
    policy_optimizer = AdamW(actor.parameters(), lr=lr)

    # Used to track steps and average reward
    tracker = {'steps': [], 'avg_reward': []}
    obs, _ = envs.reset()
    step_count = 0
    running_rewards = np.zeros(n_envs)
    recent_scores = deque(maxlen=50)
    
    print(f"[{name}] Training SAC ... LR={lr} | HIDDEN={hidden_size} | ALPHAS={alpha}")
    # Initial Random Fill which populates replay buffer with random experiences before training
    while len(replay_memory.memory) < min_replay_size :
        # Each environment gets a random action
        actions = np.array([envs.single_action_space.sample() for _ in range(n_envs)])
        # Receives next states, rewards, and done flags from all parallel envs
        next_obs, rewards, terminations, truncations, _ = envs.step(actions)
        dones = terminations | truncations
        
        # Stores each environment's transition in the replay buffer
        for i in range(n_envs):
            replay_memory.append(Transition(obs[i], actions[i], rewards[i], dones[i], next_obs[i]))
        obs = next_obs
            
    obs, _ = envs.reset()
    while step_count < max_steps:
        step_count += n_envs  # Increments by number of parallel steps
        
        with torch.no_grad():
            state_t = torch.as_tensor(obs, dtype=torch.float32).to(device)
            actions, _ = actor(state_t)
            actions = actions.cpu().numpy()

        next_obs, rewards, terminations, truncations, _ = envs.step(actions)
        dones = terminations | truncations
        
        # Stores transitions for each parallel environment
        for i in range(n_envs):
            replay_memory.append(Transition(obs[i], actions[i], rewards[i], dones[i], next_obs[i]))
        
        running_rewards += rewards
        
        # Tracks completed episodes
        for i, done in enumerate(dones):
            if done:
                recent_scores.append(running_rewards[i])
                running_rewards[i] = 0
        
        obs = next_obs

        # Logging and early stopping
        if len(recent_scores) > 0:
            avg_score = np.mean(recent_scores)
            
            if step_count % 10000 < n_envs:
                tracker['steps'].append(step_count)
                tracker['avg_reward'].append(avg_score)
                print(f"{name} Step: {step_count} | Avg Reward: {avg_score:.2f}")
            
            if len(recent_scores) == 50 and avg_score >= 195.0:
                tracker['steps'].append(step_count)
                tracker['avg_reward'].append(avg_score)
                print(f"{name} Solved at Step {step_count} with Avg Reward {avg_score:.2f}!")
                break

        # Performs multiple updates per env step for better sample efficiency
        for _ in range(n_envs):
            states, actions_b, rewards_b, dones_b, next_states = replay_memory.sample_batch()

            action_values1 = q_net1(states, actions_b)
            action_values2 = q_net2(states, actions_b)

            with torch.no_grad():
                target_actions, target_log_probs = actor(next_states)

                next_action_values = torch.min(
                    q_net1_target(next_states, target_actions),
                    q_net2_target(next_states, target_actions)
                )
                target = rewards_b + gamma * (1 - dones_b.float()) * (next_action_values - alpha * target_log_probs)

            Qloss1 = F.smooth_l1_loss(target, action_values1)
            Qloss2 = F.smooth_l1_loss(target, action_values2)
            Qloss_total = (Qloss1 + Qloss2)

            q_net_optimizer.zero_grad()
            Qloss_total.backward()
            q_net_optimizer.step()

            new_actions, log_probs = actor(states)
            action_values = torch.min(
                q_net1(states, new_actions),
                q_net2(states, new_actions)
            )
            policy_loss = (alpha * log_probs - action_values).mean()

            policy_optimizer.zero_grad()
            policy_loss.backward()
            policy_optimizer.step()

            # Updates Targets (Actor was removed since adding a target actor was not needed)
            for target_param, param in zip(q_net1_target.parameters(), q_net1.parameters()):
                target_param.data.copy_(param.data * tau  + target_param.data * (1.0 - tau))

            for target_param, param in zip(q_net2_target.parameters(), q_net2.parameters()):
                target_param.data.copy_(param.data * tau  + target_param.data * (1.0 - tau))

    envs.close()
    return tracker, actor

## Training and Hyperparameter Tuning

### PPO Grid Search

In [ ]:
def tune_ppo():
    LRS = [1e-4, 3e-4, 5e-4]
    HIDDEN = [128, 256]
    CLIPS = [0.1, 0.2]
    grid_results = {}
    best_avg_reward = -float('inf')
    best_params = {}
    best_model = None
    best_results = None

    print("Starting PPO Grid Search...")

    for lr in LRS:
        for hd in HIDDEN:
            for cp in CLIPS:
                current_params = {
                    'LR': lr,
                    'CLIP_EPS': cp,
                    'HIDDEN_SIZE': hd
                }
                run_name = f"PPO_LR{lr}_HIDDEN{hd}_CLIPS{cp}"
                print(f"\n--- Running {run_name} ---")
                results, model = train_ppo(current_params, name=run_name)

                grid_results[run_name] = results

                final_avg = results['avg_reward'][-1] if len(results['avg_reward']) > 0 else -999
                if final_avg > best_avg_reward:
                    best_avg_reward = final_avg
                    best_params = current_params
                    best_model = model
                    best_results = results
                    print(f"New Best: {run_name} (Avg: {final_avg:.2f})")

    print(f"\nPPO Grid Search Complete. Best Params: {best_params}")
    return best_params, best_model, best_results, grid_results

PPO_params, PPO_model, PPO_results, PPO_grid_results = tune_ppo()

### SAC Grid Search

In [ ]:
def tune_sac():
    LRS = [1e-4, 3e-4, 5e-4]
    HIDDEN = [64, 128]
    ALPHAS = [0.05, 0.1]
    grid_results = {}
    best_avg_reward = -float('inf')
    best_params = {}
    best_model = None
    best_results = None

    print("Starting SAC Grid Search...")

    for lr in LRS:
        for hd in HIDDEN:
            for ap in ALPHAS:
                current_params = {
                    'LR': lr,
                    'BATCH_SIZE': 256, # Batch size changes due to model requirements
                    'ALPHA': ap,
                    'HIDDEN_SIZE': hd,
                }
                run_name = f"SAC_LR{lr}_HIDDEN{hd}_ALPHA{ap}"
                print(f"\n--- Running {run_name} ---")
                results, model = train_sac(current_params, name=run_name)

                grid_results[run_name] = results

                final_avg = results['avg_reward'][-1] if len(results['avg_reward']) > 0 else -999
                if final_avg > best_avg_reward:
                    best_avg_reward = final_avg
                    best_params = current_params
                    best_model = model
                    best_results = results
                    print(f"New Best: {run_name} (Avg: {final_avg:.2f})")

    print(f"\nSAC Grid Search Complete. Best Params: {best_params}")
    return best_params, best_model, best_results, grid_results

SAC_params, SAC_model, SAC_results, SAC_grid_results = tune_sac()

## Comparisons

In [ ]:
# Function used to plot the grid search results
def grid_plot_comparison(grid_results, title):
    plt.figure(figsize=(12, 8))

    # Plots all grid search results
    for name, res in grid_results.items():
        plt.plot(res['steps'], res['avg_reward'], label=name)

    plt.xlabel('Environment Timesteps')
    plt.ylabel('Average Reward (50-ep)')
    plt.title(title)
    plt.legend()
    plt.grid(True)
    plt.show()

# Function used to plot the best models
def plot_comparison(results_1, results_2, title):
    plt.figure(figsize=(12, 8))

    # Plots PPO
    if results_1:
        plt.plot(results_1['steps'], results_1['avg_reward'], label='PPO (Best)', linewidth=3, linestyle='--', color='blue')

    # Plots SAC
    if results_2:
        plt.plot(results_2['steps'], results_2['avg_reward'], label='SAC (Best)', linewidth=3, linestyle='--', color='orange')

    plt.xlabel('Environment Timesteps')
    plt.ylabel('Average Reward (50-ep)')
    plt.title(title)
    plt.legend()
    plt.grid(True)
    plt.show()

grid_plot_comparison(PPO_grid_results, 'Grid Search of PPO')
grid_plot_comparison(SAC_grid_results, 'Grid Search of SAC')
plot_comparison(PPO_results, SAC_results, 'Best Models Comparison')

In [ ]:
# Function used to test robustness of Continuous Agents (PPO/SAC) against wind and turbulence
def test_robustness_continuous(agent, agent_type, wind_powers, turbulence_powers, num_episodes, seed=50):
    # Initialises a 2D array to store results for each combination
    results = np.zeros((len(wind_powers), len(turbulence_powers)))
    
    # Loops through each combination of wind and turbulence
    for i, w in enumerate(wind_powers):
        for j, t in enumerate(turbulence_powers):
            # Environments with wind and turbulence - note the Continuous-v3 name
            env_test = gym.make("LunarLanderContinuous-v3", enable_wind=True, wind_power=w, turbulence_power=t)
            total = 0
            for ep in range(num_episodes):
                obs, _ = env_test.reset(seed=seed if ep == 0 else None)
                done = False
                while not done:
                    obs_t = torch.as_tensor(obs, dtype=torch.float32).unsqueeze(0).to(device)
                    with torch.no_grad():
                        if agent_type == "PPO":
                            # Uses the mean (loc) for deterministic testing
                            action, _ = agent(obs_t)
                        else: # SAC
                            action, _ = agent(obs_t)
                        action = action.cpu().numpy()[0]
                        
                    obs, r, term, trunc, _ = env_test.step(action)
                    done = term or trunc
                    total += r
            results[i, j] = total / num_episodes
            env_test.close()
    return results

# Defines wind and turbulence parameters
winds = [0.0, 5.0, 10.0, 15.0, 20.0]
turbulences = [0.0, 0.5, 1.0, 1.5, 2.0]
num_test_eps = 50

print("Testing Robustness (Best PPO)...")
res_ppo = test_robustness_continuous(PPO_model, "PPO", winds, turbulences, num_test_eps)

print("\nTesting Robustness (Best SAC)...")
res_sac = test_robustness_continuous(SAC_model, "SAC", winds, turbulences, num_test_eps)

# Side-by-side plotting for report
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6), sharey=True)

# Plots results for each wind speed and turbulence
for i, w in enumerate(winds):
    ax1.plot(turbulences, res_ppo[i, :], marker='o', label=f'Wind={w}')
    ax2.plot(turbulences, res_sac[i, :], marker='s', label=f'Wind={w}')

ax1.set_title('Best PPO Robustness (Continuous)')
ax1.set_xlabel('Turbulence Power')
ax1.set_ylabel(f'Avg Reward ({num_test_eps} eps)')
ax1.legend()
ax1.grid(True)

ax2.set_title('Best SAC Robustness (Continuous)')
ax2.set_xlabel('Turbulence Power')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

## Visualisation

In [ ]:
# Makes the environment have a video renderer for PPO
if PPO_model:
    print("Visualizing Best_PPO...")
    env = gym.make("LunarLanderContinuous-v3", render_mode="rgb_array")
    env = rl.RenderFrame(env, "./video_Best_PPO")

    obs, info = env.reset(seed=50)

    while True:
        with torch.no_grad():
            obs_t = torch.as_tensor(obs, dtype=torch.float32).unsqueeze(0).to(device)
            # PPO Actor returns (loc, scale). unpacking (action, _) keeps 'loc' (mean) for deterministic play.
            action, _ = PPO_model(obs_t) 
            action = action.cpu().numpy()[0]

        obs, reward, terminated, truncated, info = env.step(action)
        if terminated or truncated:
            break

    env.play()

# Makes the environment have a video renderer for SAC
if SAC_model:
    print("Visualizing Best_SAC...")
    env = gym.make("LunarLanderContinuous-v3", render_mode="rgb_array")
    env = rl.RenderFrame(env, "./video_Best_SAC")

    obs, info = env.reset(seed=50)

    while True:
        with torch.no_grad():
            obs_t = torch.as_tensor(obs, dtype=torch.float32).unsqueeze(0).to(device)
            # SAC Policy returns (action, log_prob). unpacking (action, _) keeps the action.
            action, _ = SAC_model(obs_t)
            action = action.cpu().numpy()[0]

        obs, reward, terminated, truncated, info = env.step(action)
        if terminated or truncated:
            break

    env.play()